# Step 0: moving-object masks with SAM 3

XM starts from the masks of everything that moves in the video. For each moving object you give SAM 3 a few clicked points
or a text prompt, SAM 3 tracks it through the whole video, you check the result and save it. The first pipeline step
(`get_depths_clean.py`) stops with a warning while these masks are missing.

Written into the scene folder:
```
<scene>/masks/<object>/<frame>.png   one folder per object (white = object)
<scene>/masks/<frame>.png            all objects combined: what XM reads (rebuilt by every save)
```
Needs: the SAM 3 code (https://github.com/facebookresearch/sam3) importable in this kernel, access to the gated
`facebook/sam3` weights on Hugging Face (log in once with `hf auth login`), and a GPU (about 8 GB for 150 frames at 720 px).
Clicking points in the figure needs `ipympl`; without it, type the pixel coordinates.

In [ ]:
import os, sys

# Scene: the same variables as every XM step (or set them in the shell before starting Jupyter)
os.environ.setdefault("XM_DATASET_CAT", "Custom")        # Custom, DAVIS, Dycheck, NVIDIA, Sintel, Syn4D, ...
os.environ.setdefault("XM_DATASET_NAME", "my_scene")      # folder datasets/<CAT>/<NAME> with images/ inside
# os.environ["XM_DATASET_ROOT"] = "/path/to/scene"        # or point at a scene folder anywhere

SAM3_REPO = os.path.expanduser("~/sam3")   # the cloned SAM 3 repo (ignored if sam3 is pip-installed)
INTERACTIVE = False                        # True: click points in the figure (needs ipympl)

## Load the video and SAM 3

In [ ]:
import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

if INTERACTIVE:
    get_ipython().run_line_magic("matplotlib", "widget")
sys.path.insert(0, os.path.abspath("."))    # XM's Python/ folder (this notebook's folder)
if os.path.isdir(SAM3_REPO):
    sys.path.insert(0, SAM3_REPO)
from utils import definitions as D
from sam3.model_builder import build_sam3_video_predictor


def _numeric_key(p):                        # XM's frame order (utils.track_tapnet._numeric_key)
    s = Path(p).stem
    try:
        return int(s)
    except ValueError:
        return s


FRAMES = sorted(glob.glob(str(D.video_dir / D.image_pattern)), key=_numeric_key)
assert FRAMES, f"no frames matching {D.image_pattern} in {D.video_dir}"
T = len(FRAMES)
W, H = Image.open(FRAMES[0]).size

# SAM 3 reads a folder of <index>.<ext> frames: link XM's frames under that naming (in outputs/, the images stay untouched)
LINKS = D.output_dir / "sam3_frames"
LINKS.mkdir(parents=True, exist_ok=True)
for l in LINKS.iterdir():
    l.unlink()
for i, f in enumerate(FRAMES):
    (LINKS / f"{i:05d}{Path(f).suffix.lower()}").symlink_to(os.path.abspath(f))

torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
predictor = build_sam3_video_predictor()
session = predictor.handle_request(dict(type="start_session", resource_path=str(LINKS), offload_video_to_cpu=True))["session_id"]
print(f"{T} frames of {W} x {H} from {D.video_dir}\nmasks go to {D.mask_dir}")

In [ ]:
OBJECTS = {}        # name -> prompts and the tracked masks (T, H, W)
COLORS = plt.cm.tab10.colors


def _track(points=None, text=None, frame=0):
    """One SAM 3 run on the whole video. points: {frame: (xy list, label list)} for one object;
    text: a concept prompt (every instance becomes its own id). Returns {id: (T, H, W) bool}."""
    predictor.handle_request(dict(type="reset_session", session_id=session))
    if text is not None:
        predictor.handle_request(dict(type="add_prompt", session_id=session, frame_index=int(frame), text=text))
        start = int(frame)
    else:
        for t, (xy, lab) in sorted(points.items()):
            predictor.handle_request(dict(type="add_prompt", session_id=session, frame_index=int(t), obj_id=1,
                                          points=[[x / W, y / H] for x, y in xy], point_labels=list(lab)))
        start = min(points)
    out = {}
    for r in predictor.handle_stream_request(dict(type="propagate_in_video", session_id=session,
                                                  propagation_direction="both", start_frame_index=start)):
        o = r["outputs"]
        for oid, m in zip(np.asarray(o["out_obj_ids"]).ravel(), np.asarray(o["out_binary_masks"])):
            out.setdefault(int(oid), np.zeros((T, H, W), bool))[r["frame_index"]] = m
    return out


def track_points(name, frame, points, labels=None):
    """Track one object from points clicked on `frame`: points [(x, y), ...] in pixels, labels 1 = object, 0 = not the
    object (default: all 1). Calling it again with the same name on another frame adds those points (fix where it fails)."""
    obj = OBJECTS.get(name, {})
    prompts = obj.get("points", {})
    prompts[int(frame)] = (list(points), [1] * len(points) if labels is None else list(labels))
    masks = _track(points=prompts).get(1, np.zeros((T, H, W), bool))
    OBJECTS[name] = dict(points=prompts, masks=masks)
    preview(name)


def track_text(name, text, frame=0, keep=None):
    """Track every instance of a concept ("person", "hand", "dog", ...) found from `frame` on. All instances form one
    object unless `keep` lists the instance ids to keep (the preview labels them)."""
    inst = _track(text=text, frame=frame)
    ids = sorted(inst) if keep is None else list(keep)
    masks = np.any([inst[i] for i in ids], 0) if ids else np.zeros((T, H, W), bool)
    OBJECTS[name] = dict(text=text, instances=inst, masks=masks)
    print(f"'{text}': instances {sorted(inst)}, kept {ids}")
    preview(name)


def show(frame=0, name=None, ax=None):
    """Frame with pixel axes (to read off click coordinates), with the object's mask and points if given."""
    ax = ax or plt.subplots(figsize=(10, 10 * H / W))[1]
    ax.imshow(Image.open(FRAMES[frame]))
    if name is not None:
        obj = OBJECTS[name]
        for k, (i, m) in enumerate(sorted(obj.get("instances", {1: obj["masks"]}).items())):
            if m[frame].any():
                ax.imshow(np.dstack([m[frame]] * 3 + [m[frame] * 0.5]) * [*COLORS[k % 10], 1])
                ax.contour(m[frame], levels=[0.5], colors=[COLORS[k % 10]], linewidths=1.5)
                if "instances" in obj:
                    y, x = np.argwhere(m[frame]).mean(0)
                    ax.text(x, y, str(i), color="w", fontsize=12, ha="center", bbox=dict(fc="k", alpha=0.6))
        for (x, y), l in zip(*obj.get("points", {}).get(frame, ([], []))):
            ax.plot(x, y, "o" if l else "x", ms=9, mec="k", mfc="lime" if l else "red", color="red")
    ax.set_title(f"frame {frame}" + (f": {name}" if name else ""))
    return ax


def preview(name, n=6):
    """n evenly spaced frames with the object's mask, and on how many frames it is present."""
    m = OBJECTS[name]["masks"]
    present = m.reshape(T, -1).any(1)
    print(f"{name}: on {present.sum()} / {T} frames" + (f", missing on {np.flatnonzero(~present)[:20].tolist()}" if (~present).any() else ""))
    ts = np.linspace(0, T - 1, n).round().astype(int)
    fig, axs = plt.subplots(1, n, figsize=(3 * n, 3 * H / W + 0.4))
    for ax, t in zip(np.atleast_1d(axs), ts):
        show(t, name, ax)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def preview_video(name, fps=10):
    """Whole video with the mask, written to outputs/sam3_masks_preview/<name>.mp4 (needs opencv)."""
    import cv2
    out = D.output_dir / "sam3_masks_preview"
    out.mkdir(parents=True, exist_ok=True)
    vw = cv2.VideoWriter(str(out / f"{name}.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    for f, m in zip(FRAMES, OBJECTS[name]["masks"]):
        im = cv2.imread(f)
        im[m] = (0.5 * im[m] + [0, 0, 127]).astype(np.uint8)
        vw.write(im)
    vw.release()
    print(out / f"{name}.mp4")


class PointPicker:
    """INTERACTIVE = True only: click on the frame, left = object, right = not the object.
    Then: track_points(name, picker.frame, picker.points, picker.labels)."""
    def __init__(self, frame=0):
        self.frame, self.points, self.labels = frame, [], []
        self.ax = show(frame)
        self.ax.figure.canvas.mpl_connect("button_press_event", self._click)

    def _click(self, e):
        if e.inaxes is self.ax:
            self.points.append((e.xdata, e.ydata))
            self.labels.append(0 if e.button == 3 else 1)
            self.ax.plot(e.xdata, e.ydata, "o" if self.labels[-1] else "x", ms=9, mec="k", mfc="lime", color="red")


def save(name):
    """Write masks/<name>/ and rebuild the combined masks/<frame>.png that XM reads."""
    d = D.mask_dir / name
    d.mkdir(parents=True, exist_ok=True)
    for f, m in zip(FRAMES, OBJECTS[name]["masks"]):
        Image.fromarray(m.astype(np.uint8) * 255).save(d / f"{Path(f).stem}.png")
    combine()


def remove(name):
    """Delete a saved object folder and rebuild the combined masks."""
    import shutil
    shutil.rmtree(D.mask_dir / name)
    combine()


def combine():
    """masks/<frame>.png = union of every object folder in masks/ (also ones saved in earlier sessions)."""
    marker = D.mask_dir / ".combined_from_object_folders"
    if glob.glob(str(D.mask_dir / "*.png")) and not marker.exists():
        raise RuntimeError(f"{D.mask_dir} already holds masks that this notebook did not make. Move them into a sub-folder "
                           "(e.g. masks/old/, then they count as one object) or delete them, and save again.")
    objs = sorted(p for p in D.mask_dir.iterdir() if p.is_dir())
    for f in FRAMES:
        u = np.zeros((H, W), bool)
        for o in objs:
            p = o / f"{Path(f).stem}.png"
            if p.exists():
                u |= np.asarray(Image.open(p).convert("L")) > 0
        Image.fromarray(u.astype(np.uint8) * 255).save(D.mask_dir / f"{Path(f).stem}.png")
    marker.write_text("masks/*.png are the union of the object folders, written by make_masks_sam3.ipynb\n")
    print(f"saved; {D.mask_dir} now combines {[o.name for o in objs]}")

## Look at the video
Pick a frame where the object is clearly visible and read the pixel coordinates of a few points on it off the axes.

In [ ]:
for t in np.linspace(0, T - 1, 3).round().astype(int):
    show(t)
plt.show()

## Add an object by clicking points
One call per object. `points` are (x, y) pixels on `frame`; `labels` 1 = on the object, 0 = not the object.
Where the preview shows the object lost or leaking, call `track_points` again with the **same name** on that frame:
the new points are added to the old ones and the whole video is tracked again.

With `INTERACTIVE = True`, click instead of typing: run `picker = PointPicker(frame)`, click, then
`track_points("my_object", picker.frame, picker.points, picker.labels)`.

In [ ]:
# edit: name, frame and the points on the object
track_points("object_1", frame=0, points=[(W // 2, H // 2)])

## Add an object by a text prompt
SAM 3 finds and tracks every instance of the concept. All instances form one object; to keep only some, pass
`keep=[ids]` with the ids shown in the preview (e.g. `track_text("left_hand", "hand", keep=[0])`).

In [ ]:
# edit: name and prompt
track_text("people", "person")

## Check and save
`preview_video(name)` writes the whole tracked video for checking. `save(name)` writes `masks/<name>/` and the combined
masks; `remove(name)` deletes a saved object again.

In [ ]:
for name in OBJECTS:
    save(name)

## Done
Close the session to free the GPU. Then run XM as usual; `python check_masks.py` confirms that the masks are found.

In [ ]:
predictor.handle_request(dict(type="close_session", session_id=session))